In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import optuna

In [2]:
# 1. Load and Prepare Data (Same as before)
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

for df in [train_df, test_df]:
    df['u_minus_g'] = df['u'] - df['g']
    df['g_minus_r'] = df['g'] - df['r']
    df['r_minus_i'] = df['r'] - df['i']
    df['i_minus_z'] = df['i'] - df['z']
    df['total_mag'] = df['u'] + df['g'] + df['r'] + df['i'] + df['z']

X_train = train_df.drop(['id', 'class'], axis=1)
y_train = train_df['class']
X_test = test_df.drop(['id'], axis=1)

cat_cols = ['spectral_type', 'galaxy_population']
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

In [3]:
# 2. Define the Objective Function for Optuna
def objective(trial):
    param = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss', # Optuna optimizes this, but we check Balanced Acc manually
        'verbose': -1,
        'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**param, n_estimators=500)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        score = balanced_accuracy_score(y_val, preds)
        scores.append(score)
        
    return np.mean(scores)

In [4]:
# 3. Run the Study
print("--- Starting Optuna Hyperparameter Tuning (50 Trials) ---")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\n*** Best Trial Balanced Accuracy: {study.best_value:.4f} ***")
print("Best Parameters:", study.best_params)


[I 2026-06-24 08:37:33,361] A new study created in memory with name: no-name-f37e7943-dbba-4d50-98b1-3055dc29d561


--- Starting Optuna Hyperparameter Tuning (50 Trials) ---


[I 2026-06-24 08:38:53,685] Trial 0 finished with value: 0.9560629731206133 and parameters: {'learning_rate': 0.1552955150611843, 'num_leaves': 117, 'max_depth': 6, 'reg_alpha': 1.171872692034255, 'reg_lambda': 1.7542439740442028e-06, 'min_child_samples': 19, 'subsample': 0.8591622830131846, 'colsample_bytree': 0.8309048974574993}. Best is trial 0 with value: 0.9560629731206133.
[I 2026-06-24 08:40:05,789] Trial 1 finished with value: 0.9551352608310939 and parameters: {'learning_rate': 0.05216166731515515, 'num_leaves': 40, 'max_depth': 7, 'reg_alpha': 0.007797274427841566, 'reg_lambda': 0.3565370788547473, 'min_child_samples': 76, 'subsample': 0.9405352529888875, 'colsample_bytree': 0.6907679520122978}. Best is trial 0 with value: 0.9560629731206133.
[I 2026-06-24 08:41:12,805] Trial 2 finished with value: 0.9444604635699317 and parameters: {'learning_rate': 0.22398648234633098, 'num_leaves': 55, 'max_depth': 5, 'reg_alpha': 0.029337738271376965, 'reg_lambda': 1.0816649339631842e-06,


*** Best Trial Balanced Accuracy: 0.9565 ***
Best Parameters: {'learning_rate': 0.10468466935077643, 'num_leaves': 114, 'max_depth': 8, 'reg_alpha': 0.023271369325013745, 'reg_lambda': 0.5558612550390504, 'min_child_samples': 72, 'subsample': 0.7215729565534106, 'colsample_bytree': 0.7981024093195654}


In [5]:
# 4. Train Final Model with Best Params
best_params = study.best_params
final_model = lgb.LGBMClassifier(**best_params, n_estimators=1000, verbose=-1, n_jobs=-1)
final_model.fit(X_train, y_train)

test_preds = final_model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'class': test_preds
})

submission.to_csv('submission_v3_tuned.csv', index=False)
print("Success! submission_v3_tuned.csv is ready.")

Success! submission_v3_tuned.csv is ready.
